In [13]:
import numpy as np
import tensorflow as tf
from typing import Any, Callable, Dict, List, Mapping, Optional, Tuple, Union

# Type annotations.
States = Dict[str, tf.Tensor]
Activation = Union[str, Callable]

In [86]:
@tf.keras.utils.register_keras_serializable(package='Vision')
class Scale(tf.keras.layers.Layer):
    """Scales the input by a trainable scalar weight.

    This is useful for applying ReZero to layers, which improves convergence
    speed. This implements the paper:
    ReZero is All You Need: Fast Convergence at Large Depth.
    (https://arxiv.org/pdf/2003.04887.pdf).
    """

    def __init__(
        self,
        initializer: tf.keras.initializers.Initializer = 'ones',
        regularizer: Optional[tf.keras.regularizers.Regularizer] = None,
        **kwargs):
        """Initializes a scale layer.

        Args:
          initializer: A `str` of initializer for the scalar weight.
          regularizer: A `tf.keras.regularizers.Regularizer` for the scalar weight.
          **kwargs: Additional keyword arguments to be passed to this layer.

        Returns:
          An `tf.Tensor` of which should have the same shape as input.
        """
        super(Scale, self).__init__(**kwargs)

        self._initializer = initializer
        self._regularizer = regularizer

        self._scale = self.add_weight(
            name='scale',
            shape=[],
            dtype=self.dtype,
            initializer=self._initializer,
            regularizer=self._regularizer,
            trainable=True)

    def get_config(self):
        """Returns a dictionary containing the config used for initialization."""
        config = {
            'initializer': self._initializer,
            'regularizer': self._regularizer,
        }
        base_config = super(Scale, self).get_config()
        return dict(list(base_config.items()) + list(config.items()))

    def call(self, inputs):
        """Calls the layer with the given inputs."""
        scale = tf.cast(self._scale, inputs.dtype)
        return scale * inputs



@tf.keras.utils.register_keras_serializable(package='Vision')
class PositionalEncoding(tf.keras.layers.Layer):
    """Creates a network layer that adds a sinusoidal positional encoding.

    Positional encoding is incremented across frames, and is added to the input.
    The positional encoding is first weighted at 0 so that the network can choose
    to ignore it. This implements:

    Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones,
    Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin.
    Attention Is All You Need.
    (https://arxiv.org/pdf/1706.03762.pdf).
    """

    def __init__(self,
                initializer: tf.keras.initializers.Initializer = 'zeros',
                cache_encoding: bool = False,
                state_prefix: Optional[str] = None,
                **kwargs):
        """Initializes positional encoding.

        Args:
          initializer: A `str` of initializer for weighting the positional encoding.
          cache_encoding: A `bool`. If True, cache the positional encoding tensor
            after calling build. Otherwise, rebuild the tensor for every call.
            Setting this to False can be useful when we want to input a variable
            number of frames, so the positional encoding tensor can change shape.
          state_prefix: a prefix string to identify states.
          **kwargs: Additional keyword arguments to be passed to this layer.

        Returns:
          A `tf.Tensor` of which should have the same shape as input.
        """
        super(PositionalEncoding, self).__init__(**kwargs)
        self._initializer = initializer
        self._cache_encoding = cache_encoding
        self._pos_encoding = None
        self._rezero = Scale(initializer=initializer, name='rezero')
        state_prefix = state_prefix if state_prefix is not None else ''
        self._state_prefix = state_prefix
        self._frame_count_name = f'{state_prefix}_pos_enc_frame_count'

    def get_config(self):
        """Returns a dictionary containing the config used for initialization."""
        config = {
            'initializer': self._initializer,
            'cache_encoding': self._cache_encoding,
            'state_prefix': self._state_prefix,
        }
        base_config = super(PositionalEncoding, self).get_config()
        return dict(list(base_config.items()) + list(config.items()))

    def _positional_encoding(self,
                            num_positions: Union[int, tf.Tensor],
                            hidden_size: Union[int, tf.Tensor],
                            start_position: Union[int, tf.Tensor] = 0,
                            dtype: str = 'float32') -> tf.Tensor:
        """Creates a sequence of sinusoidal positional encoding vectors.

        Args:
          num_positions: the total number of positions (frames).
          hidden_size: the number of channels used for the hidden vectors.
          start_position: the start position.
          dtype: the dtype of the output tensor.

        Returns:
          The positional encoding tensor with shape [num_positions, hidden_size].
        """
        if isinstance(start_position, tf.Tensor) and start_position.shape.rank == 1:
          start_position = start_position[0]

        # Calling `tf.range` with `dtype=tf.bfloat16` results in an error,
        # so we cast afterward.
        positions = tf.range(start_position, start_position + num_positions)
        positions = tf.cast(positions, dtype)[:, tf.newaxis]
        idx = tf.range(hidden_size)[tf.newaxis, :]

        power = tf.cast(2 * (idx // 2), dtype)
        power /= tf.cast(hidden_size, dtype)
        angles = 1. / tf.math.pow(10_000., power)
        radians = positions * angles

        sin = tf.math.sin(radians[:, 0::2])
        cos = tf.math.cos(radians[:, 1::2])
        pos_encoding = tf.concat([sin, cos], axis=-1)

        return pos_encoding

    def _get_pos_encoding(self,
                          input_shape: tf.Tensor,
                          frame_count: int = 0) -> tf.Tensor:
        """Calculates the positional encoding from the input shape.

        Args:
          input_shape: the shape of the input.
          frame_count: a count of frames that indicates the index of the first
            frame.

        Returns:
          The positional encoding tensor with shape [num_positions, hidden_size].

        """
        frames = input_shape[1]
        channels = input_shape[-1]
        pos_encoding = self._positional_encoding(
            frames, channels, start_position=frame_count, dtype=self.dtype)
        pos_encoding = tf.reshape(pos_encoding, [1, frames, channels])
        return pos_encoding

    def build(self, input_shape):
        """Builds the layer with the given input shape.

        Args:
          input_shape: The input shape.

        Raises:
          ValueError: If using 'channels_first' data format.
        """
        if tf.keras.backend.image_data_format() == 'channels_first':
            raise ValueError('"channels_first" mode is unsupported.')

        if self._cache_encoding:
            self._pos_encoding = self._get_pos_encoding(input_shape)

        super(PositionalEncoding, self).build(input_shape)

    def call(
        self,
        inputs: tf.Tensor,
        states: Optional[States] = None,
        output_states: bool = False,
      ) -> Union[tf.Tensor, Tuple[tf.Tensor, States]]:
        """Calls the layer with the given inputs.

        Args:
          inputs: An input `tf.Tensor`.
          states: A `dict` of states such that, if any of the keys match for this
            layer, will overwrite the contents of the buffer(s). Expected keys
            include `state_prefix + '_pos_enc_frame_count'`.
          output_states: A `bool`. If True, returns the output tensor and output
            states. Returns just the output tensor otherwise.

        Returns:
          An output `tf.Tensor` (and optionally the states if `output_states=True`).

        Raises:
          ValueError: If using 'channels_first' data format.
        """
        states = dict(states) if states is not None else {}

        # Keep a count of frames encountered across input iterations in
        # num_frames to be able to accurately update the positional encoding.
        num_frames = tf.shape(inputs)[1]
        frame_count = tf.cast(states.get(self._frame_count_name, [0]), tf.int32)
        states[self._frame_count_name] = frame_count + num_frames

        if self._cache_encoding:
            pos_encoding = self._pos_encoding
        else:
            pos_encoding = self._get_pos_encoding(
              tf.shape(inputs), frame_count=frame_count)
        pos_encoding = tf.cast(pos_encoding, inputs.dtype)
        pos_encoding = self._rezero(pos_encoding)

        outputs = inputs + pos_encoding
        return (outputs, states) if output_states else outputs

In [ ]:
class CustomizedMultiheadAttention(tf.keras.layers.Layer):
    def __init__(self, H, d_model, dk, dv):    
        """
        Arguments:
        H -- number of heads (=8 in the paper)
        d_models -- embedding dimension (=512 in the paper)
        dk -- depth of Q and K (=64 in the paper)
        dv -- depth of V (=64 in the paper)
        """
    
        super(CustomizedMultiheadAttention, self).__init__()
        
        initializer = tf.keras.initializers.GlorotUniform()
        self.WQ = tf.Variable(initializer(shape=(H, d_model, dk)), trainable=True)
        self.WK = tf.Variable(initializer(shape=(H, d_model, dk)), trainable=True)
        self.WV = tf.Variable(initializer(shape=(H, d_model, dv)), trainable=True)
        self.WO = tf.Variable(initializer(shape=(H*dv, d_model)), trainable=True)

    
    def call(self, Q, K, V, mask=None):
        """
        Calculate the attention weights.

        Arguments:
            Q -- query shape == (..., Tq, d_model)
            K -- key shape == (..., Tv, d_model)
            V -- value shape == (..., Tv, d_model)
            mask: Float tensor with shape broadcastable to (..., Tq, Tv). Defaults to None.

        Returns:
            output -- Multihead attention A of shape (batch_size, Tq, d_model)
        """
        #   Projecting Q,K,V to Qh, Kh, Vh. The H projection are stacked on the penultiem axis
        Qh = tf.experimental.numpy.dot(Q, self.WQ) #of shape (batch_size, Tq, H, dk)
        Kh = tf.experimental.numpy.dot(K, self.WK) #of shape (batch_size, Tv, H, dk)
        Vh = tf.experimental.numpy.dot(V, self.WV) #of shape (batch_size, Tv, H, dv)
        
        #   Transposition
        Qh = tf.transpose(Qh, [0,2,1,3]) #of shape (batch_size, H, Tq, dk)
        Kh = tf.transpose(Kh, [0,2,1,3]) #of shape (batch_size, H, Tv, dk)
        Vh = tf.transpose(Vh, [0,2,1,3]) #of shape (batch_size, H, Tv, dv)
        
        #   Computing the dot-product attention
        Ah , _ = self.scaled_dot_product_attention(Qh, Kh, Vh, mask=mask) #of shape (batch_size, H, Tq, dv)
        
        #   Flattening the H and dv axis and projecting back to d_model
        #   A = tf.reshape(Ah,(*Ah.shape[:-2],-1))
        s = Ah.shape
        A = tf.reshape(Ah,(s[0],s[2],s[1]*s[3])) #of shape (batch_size, Tq, H*dv)
        A = tf.experimental.numpy.dot(A, self.WO) #of shape (batch_size, Tq, d_model)
        return A

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Calculate the attention weights.
        Arguments:
            Q -- query shape == (..., Tq, dk)
            K -- key shape == (..., Tv, dk)
            V -- value shape == (..., Tv, dv)
            mask: Float tensor with shape broadcastable to (..., Tq, Tv). Defaults to None.
        Returns:
            output -- (attention,attention_weights)
        """
        #   Compute the scaled dot-product Q•K
        matmul_QK = tf.matmul(Q,K,transpose_b=True)  # dot-product of shape (..., Tq, Tv)
        dk = K.shape[-1]
        scaled_attention_logits = matmul_QK/np.sqrt(dk) # scaled dot-product of shape (..., Tq, Tv)
        #   Add the mask to the scaled dot-product
        if mask is not None: 
            scaled_attention_logits += (1. - mask) *(-1e9)
        # Compute the Softmax
        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)  # weights of shape (..., Tq, Tv)
        # Multiply with V
        output = tf.matmul(attention_weights,V)  # Attention representation of shape (..., Tq, dv)
        return output, attention_weights

In [87]:
class BaseAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__()
        self.mha = tf.keras.layers.MultiHeadAttention(**kwargs)
        self.layernorm = tf.keras.layers.LayerNormalization()
        self.add = tf.keras.layers.Add()


class CrossAttention(BaseAttention):
    def call(self, x, context):
        attn_output, attn_scores = self.mha(
        query=x,
        key=context,
        value=context,
        return_attention_scores=True)

        # Cache the attention scores for plotting later.
        self.last_attn_scores = attn_scores

        x = self.add([x, attn_output])
        x = self.layernorm(x)
        return x
  

class GlobalSelfAttention(BaseAttention):
    def call(self, x):
        attn_output = self.mha(
        query=x,
        value=x,
        key=x)
        x = self.add([x, attn_output])
        x = self.layernorm(x)
        return x

In [88]:
class FeedForward(tf.keras.layers.Layer):
    def __init__(self, d_model, dff, dropout_rate=0.1):
        super().__init__()
        self.seq = tf.keras.Sequential([
        tf.keras.layers.Dense(dff, activation='relu'),
        tf.keras.layers.Dense(d_model),
        tf.keras.layers.Dropout(dropout_rate)
        ])
        self.add = tf.keras.layers.Add()
        self.layer_norm = tf.keras.layers.LayerNormalization()

    def call(self, x):
        x = self.add([x, self.seq(x)])
        x = self.layer_norm(x) 
        return x

In [89]:
class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self,*, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        self.self_attention = GlobalSelfAttention(
        num_heads=num_heads,
        key_dim=d_model,
        dropout=dropout_rate)
        self.ffn = FeedForward(d_model, dff)

    def call(self, x):
        x = self.self_attention(x)
        x = self.ffn(x)
        return x


class Encoder(tf.keras.layers.Layer):
    def __init__(self, *, num_layers, d_model, num_heads,
            dff, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_layers = num_layers

        self.pos_embedding = PositionalEncoding()

        self.enc_layers = [
        EncoderLayer(d_model=d_model,
                        num_heads=num_heads,
                        dff=dff,
                        dropout_rate=dropout_rate)
        for _ in range(num_layers)]
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x):
        # `x` is token-IDs shape: (batch, seq_len)
        x = self.pos_embedding(x)  # Shape `(batch_size, seq_len, d_model)`.
        # Add dropout.
        x = self.dropout(x)
        for i in range(self.num_layers):
            x = self.enc_layers[i](x)
        return x  # Shape `(batch_size, seq_len, d_model)`.

In [90]:
class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self,
                *,
                d_model,
                num_heads,
                dff,
                dropout_rate=0.1):
        super(DecoderLayer, self).__init__()

        self.causal_self_attention = GlobalSelfAttention(
            num_heads=num_heads,
            key_dim=d_model,
            dropout=dropout_rate)

        self.cross_attention = CrossAttention(
            num_heads=num_heads,
            key_dim=d_model,
            dropout=dropout_rate)

        self.ffn = FeedForward(d_model, dff)

    def call(self, x, context):
        x = self.causal_self_attention(x=x)
        x = self.cross_attention(x=x, context=context)
        # Cache the last attention scores for plotting later
        self.last_attn_scores = self.cross_attention.last_attn_scores
        x = self.ffn(x)  # Shape `(batch_size, seq_len, d_model)`.
        return x

In [91]:
class Decoder(tf.keras.layers.Layer):
    def __init__(self, *, num_layers, d_model, num_heads, dff,
                dropout_rate=0.1):
        super(Decoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.pos_embedding = PositionalEncoding()
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        self.dec_layers = [
            DecoderLayer(d_model=d_model, num_heads=num_heads,
                        dff=dff, dropout_rate=dropout_rate)
            for _ in range(num_layers)]

        self.last_attn_scores = None

    def call(self, x, context):
        # `x` is token-IDs shape (batch, target_seq_len)
        x = self.pos_embedding(x)  # (batch_size, target_seq_len, d_model)
        x = self.dropout(x)
        for i in range(self.num_layers):
            x  = self.dec_layers[i](x, context)
        self.last_attn_scores = self.dec_layers[-1].last_attn_scores
        # The shape of x is (batch_size, target_seq_len, d_model).
        return x

In [96]:
class Transformer(tf.keras.Model):
    def __init__(self, *, num_layers, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        self.encoder = Encoder(num_layers=num_layers, d_model=d_model,
                            num_heads=num_heads, dff=dff,
                            dropout_rate=dropout_rate)

        self.decoder = Decoder(num_layers=num_layers, d_model=d_model,
                            num_heads=num_heads, dff=dff,
                            dropout_rate=dropout_rate)

        self.final_layer = tf.keras.layers.Dense(64)

    def call(self, search, template):
        # To use a Keras model with `.fit` you must pass all your inputs in the
        # first argument.
        memory = self.encoder(search)  # (batch_size, context_len, d_model)
        x = self.decoder(memory, template)  # (batch_size, target_len, d_model)
        # Final linear layer output.
        logits = self.final_layer(x)  # (batch_size, target_len, target_vocab_size)
        return logits

In [100]:
model = Transformer(
    num_layers=1,
    d_model=256,
    num_heads=8,
    dff=256,
    dropout_rate=0.1)

In [101]:
template = tf.random.uniform((1, 15 * 15, 256))
search = tf.random.uniform((1, 31 * 31, 256))

output = model(search, template)

# # Print the shape.
# print(pt.shape)
# print(sample_encoder_output.shape) 

In [102]:
model.summary()

Model: "transformer_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 encoder_19 (Encoder)        multiple                  2236161   
                                                                 
 decoder_6 (Decoder)         multiple                  4340225   
                                                                 
 dense_110 (Dense)           multiple                  16448     
                                                                 
Total params: 6592834 (25.15 MB)
Trainable params: 6592834 (25.15 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
